# AbstractTensor — Manifolds and Tensors

This notebook tests and explains how to use objects defined in `manifolds.jl`, `indices.jl` and `tensors.jl` step by step.

---
## 1. Loading package

In [1]:
using SymbolicTensors

---
## 2. Manifolds and Vbundles 
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of vector bundles with manifold as base manifold


In [52]:
?@def_manifold

```julia
@def_manifold name dim coord_indices basis_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frame bundles.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

Bind in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)
  * `frame<name>`, `coframe<name>`, moving basis symbols (default `e`, `θ`)

Coordinate indices (first list) → [`CoordinateIndex`](@ref):

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Basis indices (second list) → [`BasisIndex`](@ref):

```julia
A1          # BasisIndex(:A1, :tangentM)   — contravariant
-A1         # BasisIndex(:A1, :cotangentM) — covariant
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_manifold M d [b1, b2, b3, b4] [B1, B2, B3, B4]   # parametric dimension
```


In [4]:
@def_manifold M 2 [a1, a2, a3, a4]

LoadError: LoadError: MethodError: no method matching var"@def_manifold"(::LineNumberNode, ::Module, ::Symbol, ::Int64, ::Expr)
The function `@def_manifold` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  var"@def_manifold"(::LineNumberNode, ::Module, ::Any, ::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mmanifolds.jl:372[24m[39m

in expression starting at In[4]:1

In [5]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

LoadError: UndefVarError: `M` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [6]:
@def_manifold N d [b1, b2, b3, b4]

LoadError: LoadError: MethodError: no method matching var"@def_manifold"(::LineNumberNode, ::Module, ::Symbol, ::Symbol, ::Expr)
The function `@def_manifold` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  var"@def_manifold"(::LineNumberNode, ::Module, ::Any, ::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mmanifolds.jl:372[24m[39m

in expression starting at In[6]:1

In [7]:
# Registries
display(_MANIFOLDS)
display(_VBUNDLES)
display(_INDICES)

Dict{Symbol, Manifold}()

Dict{Symbol, VBundle}()

LoadError: UndefVarError: `_INDICES` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [8]:
M

LoadError: UndefVarError: `M` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
@undef_manifold M

LoadError: @undef_manifold: manifold M is not registered. Call @def_manifold M first, or check keys(_MANIFOLDS).

In [10]:
M.name
M.dim
M.tangent_bundle
M.cotangent_bundle
M.vbundles

LoadError: UndefVarError: `M` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
@def_manifold M 4 [a1, a2, a3, a4]

LoadError: LoadError: MethodError: no method matching var"@def_manifold"(::LineNumberNode, ::Module, ::Symbol, ::Int64, ::Expr)
The function `@def_manifold` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  var"@def_manifold"(::LineNumberNode, ::Module, ::Any, ::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mmanifolds.jl:372[24m[39m

in expression starting at In[11]:1

In [12]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

LoadError: UndefVarError: `M` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### b. Tangent and Cotangent bundles

In [13]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name      # :tangentM
tangentM.manifold  # :M
tangentM.dim       # 4
tangentM.isdual    # false
tangentM.dual      # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.basis_indices       # [BasisIndex(:A1, :tangentM), ...]
tangentM.bases     # [Basis(:∂, :tangentM, :coordinate), Basis(:e, :tangentM, :frame)]
```

### Fields

  * `name`     : bundle name, e.g. `:tangentM`
  * `manifold` : base manifold name, e.g. `:M`
  * `dim`      : fibre dimension
  * `isdual`   : false = contravariant (upper) slots, true = covariant (lower) slots.              Authoritative for index variance via [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`     : name of the paired dual bundle, e.g. `:cotangentM` or `:dualE`
  * `coordinate_indices` : [`CoordinateIndex`](@ref) for the coordinate chart (∂ / `dx`);              nonempty on tangent/cotangent bundles from [`@def_manifold`](@ref)
  * `basis_indices` : [`BasisIndex`](@ref) for fibre / moving bases (`e` / `θ`);              populated on tangent/cotangent by [`@def_manifold`](@ref) and on custom              bundles by [`@def_vbundle`](@ref)


In [14]:
tangentM

LoadError: UndefVarError: `tangentM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
display(tangentM.name)
display(tangentM.manifold)
display(tangentM.dim)
display(tangentM.isdual)
display(tangentM.indices)

LoadError: UndefVarError: `tangentM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
cotangentM

LoadError: UndefVarError: `cotangentM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
display(cotangentM.name)
display(cotangentM.manifold)
display(cotangentM.dim)
display(cotangentM.isdual)
display(cotangentM.indices)

LoadError: UndefVarError: `cotangentM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
cotangentM.indices

LoadError: UndefVarError: `cotangentM` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### c. Definition of extra Vbundles

In [19]:
?@def_vbundle

```julia
@def_vbundle name manifold dim [idx1, idx2, ...]
```

Define a new vector bundle `name` of fibre dimension `dim` over `manifold`, and its dual bundle `dual<name>`. Bind the following variables in the caller's scope:

  * `name`       → a [`VBundle`](@ref) instance (`isdual = false`)
  * `dualname`   → a [`VBundle`](@ref) instance (`isdual = true`)

Each index symbol in `[idx1, idx2, ...]` is bound to a contravariant [`BasisIndex`](@ref) and registered to `name` (the primal bundle). The dual bundle shares the same index symbols — `-A1` resolves to `BasisIndex(:A1, :dualname)` via [`flip`](@ref) and the `dual` field on [`VBundle`](@ref).

`dim` accepts a concrete integer or a bare symbol for parametric bundles. The fibre dimension is independent of the base manifold dimension.

Registers both bundles in `_VBUNDLES` and appends their names to `manifold.vbundles`.

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [B1, B2, B3, B4]
@def_vbundle E M 3 [A1, A2, A3]    # rank-3 bundle and its dual over M
@def_vbundle E M r [A1, A2, A3]    # parametric fibre dimension

E.isdual       # false
dualE.isdual   # true
A1.vbundle     # :E
-A1            # BasisIndex(:A1, :dualE)
M.vbundles     # [:tangentM, :cotangentM, :E, :dualE]
```


In [20]:
@def_vbundle E M 3 [A1, A2, A3]

LoadError: @def_vbundle: manifold M is not registered. Call @def_manifold M first.

In [21]:
E

LoadError: UndefVarError: `E` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [22]:
dualE

LoadError: UndefVarError: `dualE` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## 3. Indices and tensors

### a. Indices

In [23]:
@doc TensorIndex

No documentation found.

Binding `TensorIndex` does not exist.


In [24]:
TensorIndex(:a1,:tangentM)

LoadError: UndefVarError: `TensorIndex` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [25]:
a1

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [26]:
+a1

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [27]:
[+a1,-a2]

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [28]:
[a1,-a2]

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [29]:
flip(a1)

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [30]:
A1

LoadError: UndefVarError: `A1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [31]:
[A1,A2,-A3]

LoadError: UndefVarError: `A1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [32]:
[-A1,+a2,-a3]

LoadError: UndefVarError: `A1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### b. Tensors

In [33]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.


In [34]:
@doc @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as=:...] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All VBundle symbols must belong to the same manifold.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g M

@def_tensor T [cotangentM, cotangentM]                 # (0,2) tensor, metric auto-resolved
@def_tensor F [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T [tangentM, cotangentM]            # (1,1) mixed tensor
```


In [35]:
@def_metric g[-a1, -a2] M

LoadError: LoadError: @def_metric: first argument must be a symbol, got: g[-a1, -a2]
in expression starting at In[35]:1

In [36]:
g

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [37]:
g[-a1,-a2]
typeof(ans)

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [38]:
@doc TensorExpression

```julia
TensorExpression
```

A [`Tensor`](@ref) (definition-level schema) applied to a concrete list of [`AbstractIndex`](@ref) objects, representing one occurrence in an algebraic expression.

Constructed via `getindex` on a [`Tensor`](@ref):

```julia
F[down(a1), down(a2)]   # explicit AbstractIndex arguments
F[-a1, -a2]             # sugar: -a1 = flip(a1) via Base.:- on AbstractIndex
F[a1, -a2]              # mixed: a1 contravariant, -a2 covariant
g[a1, a2]               # valid: contravariant metric expression
```

**Validation at construction time:**

  * `length(idxs) == T.rank`
  * *Manifold membership*: each index's home tangent bundle must match `T.manifold`

**Not validated:**

  * *Variance* (up vs down) against `T.slots`. The canonical slot structure is metadata for index raising/lowering, not a construction gate.

The expression is **lazy and inert**: no contraction, canonicalization, or symmetry reduction is performed at construction time.

### Fields

  * `tensor`  : the [`Tensor`](@ref) this expression refers to
  * `indices` : the concrete index list for this occurrence, one per slot


In [39]:
g[-a1,-a2] 

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [40]:
@show g[-a1,-a2] ;

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [41]:
g[a1,-a2] 

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [42]:
g(dx_a1,dx_a2)

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [43]:
expand(g)

LoadError: UndefVarError: `expand` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [44]:
g[-a1,-a2] dx^a1 tens dx^a2

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [45]:
@def_tensor T[-a1, -a2] M 

LoadError: LoadError: @def_tensor: first argument must be a tensor name symbol, got: T[-a1, -a2]
in expression starting at In[45]:1

In [46]:
T[-a1, -a2]

LoadError: UndefVarError: `T` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [47]:
a1

LoadError: UndefVarError: `a1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [48]:
?a1

search: as

Couldn't find a1
Perhaps you meant as, Val, cat, all, any, NaN, map, max, ans, tan, abs, <: or ∈


No documentation found.

Binding `a1` does not exist.
